In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak
from tqdm import tqdm
from datetime import datetime, timedelta
from collections import Counter
from quant_utils import filter_stock,filter_main_board,filter_main_board_no_st,ts_code
import tushare as ts

In [7]:
#抓取数据（每天凌晨12：00点自动化抓取）
stock_name = ak.stock_info_a_code_name()

stock_name

  0%|          | 0/15 [00:00<?, ?it/s]

,code,name
0,000001,平安银行
1,000002,万 科Ａ
2,000004,*ST国华
3,000006,深振业Ａ
4,000007,全新好
...,...,...
5480,920978,开特股份
5481,920981,晶赛科技
5482,920982,锦波生物
5483,920985,海泰新能


In [9]:
stock_name

#加板块column
stock_name = filter_stock(stock_name)

#只要主板
main_board = filter_main_board(stock_name)

#过滤ST和*ST
no_ST = filter_main_board_no_st(main_board)

stock_name.to_csv("data/all_stock_name.csv", index=False, encoding="utf-8-sig")
main_board.to_csv("data/main_board.csv", index=False, encoding="utf-8-sig")
no_ST.to_csv("data/no_ST_main_board.csv", index=False, encoding="utf-8-sig")



In [10]:
import tushare as ts
ts_name = ts_code(no_ST)
ts_name.to_csv("data/with_ts_code.csv", index=False, encoding="utf-8-sig")


In [11]:
ts_name

,code,name,board,ts_code
0,000001,平安银行,深主板,000001.SZ
1,000002,万 科Ａ,深主板,000002.SZ
3,000006,深振业Ａ,深主板,000006.SZ
4,000007,全新好,深主板,000007.SZ
5,000008,神州高铁,深主板,000008.SZ
...,...,...,...,...
4581,605580,恒盛能源,沪主板,605580.SH
4582,605588,冠石科技,沪主板,605588.SH
4583,605589,圣泉集团,沪主板,605589.SH
4584,605598,上海港湾,沪主板,605598.SH


In [12]:
from pathlib import Path

import pandas as pd
import tushare as ts
from tqdm.auto import tqdm

ts.set_token("5b9d1667c883b96aeeb92f7f19108e95dd8b5e0dd0e33aedc6d17a68")
pro = ts.pro_api()

BY_DATE_DIR = Path("data/by_date")
BY_STOCK_DIR = Path("data/by_stock")

BY_DATE_DIR.mkdir(parents=True, exist_ok=True)
BY_STOCK_DIR.mkdir(parents=True, exist_ok=True)





In [13]:


def fetch_one_day_all_stocks_fast(stock_df, trade_date):
    """
    一次性抓取某一天全市场 daily + adj_factor，再和 stock_df 合并
    stock_df 需要包含: code, name, board, ts_code
    trade_date 格式: YYYYMMDD
    """
    daily_df = pro.daily(trade_date=trade_date)
    adj_df = pro.adj_factor(trade_date=trade_date)

    if daily_df is None or daily_df.empty:
        return pd.DataFrame()
    if adj_df is None or adj_df.empty:
        return pd.DataFrame()

    df = daily_df.merge(
        adj_df[["ts_code", "trade_date", "adj_factor"]],
        on=["ts_code", "trade_date"],
        how="left",
    )

    stock_info = stock_df.copy()
    stock_info["code"] = stock_info["code"].astype(str).str.zfill(6)

    df = df.merge(
        stock_info[["code", "name", "board", "ts_code"]],
        on="ts_code",
        how="inner",
    )

    df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["trade_date"])
    df = df.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)
    return df


def update_one_day_store(one_day_df):
    if one_day_df is None or one_day_df.empty:
        print("当天没有数据可更新")
        return

    trade_date_str = one_day_df["trade_date"].iloc[0].strftime("%Y-%m-%d")

    # 按日期存
    date_path = BY_DATE_DIR / f"{trade_date_str}.parquet"
    one_day_df.to_parquet(date_path, index=False)

    # 按股票增量存
    for ts_code, sub_df in one_day_df.groupby("ts_code"):
        stock_path = BY_STOCK_DIR / f"{ts_code}.parquet"

        if stock_path.exists():
            old_df = pd.read_parquet(stock_path)
            merged = pd.concat([old_df, sub_df], ignore_index=True)
            merged = (
                merged.sort_values("trade_date")
                .drop_duplicates(subset=["trade_date"], keep="last")
                .reset_index(drop=True)
            )
        else:
            merged = (
                sub_df.sort_values("trade_date")
                .drop_duplicates(subset=["trade_date"], keep="last")
                .reset_index(drop=True)
            )

        merged.to_parquet(stock_path, index=False)


def run_daily_update_fast(stock_df, trade_date):
    one_day_df = fetch_one_day_all_stocks_fast(stock_df, trade_date)
    update_one_day_store(one_day_df)
    return one_day_df


In [33]:
today_df = run_daily_update_fast(ts_name, trade_date="20260224")
print(today_df.shape)
display(today_df.head())


(3061, 15)


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor,code,name,board
0,000001.SZ,2026-02-24,10.93,10.95,10.88,10.91,10.91,0.00,0.0000,602512.40,657487.700,134.5794,000001,平安银行,深主板
1,000002.SZ,2026-02-24,4.94,4.96,4.90,4.92,4.97,-0.05,-1.0060,1557018.85,766341.374,181.7040,000002,万 科Ａ,深主板
2,000006.SZ,2026-02-24,9.60,9.60,9.20,9.27,9.43,-0.16,-1.6967,310211.83,287748.142,39.7400,000006,深振业Ａ,深主板
3,000007.SZ,2026-02-24,12.61,12.95,12.50,12.70,12.41,0.29,2.3368,82319.03,104592.090,8.2840,000007,全新好,深主板
4,000008.SZ,2026-02-24,2.99,3.05,2.99,3.03,2.96,0.07,2.3649,480310.00,145591.085,22.4080,000008,神州高铁,深主板


In [34]:
expected = set(ts_name["ts_code"])
actual = set(today_df["ts_code"])
missing = sorted(expected - actual)

missing_df = ts_name[ts_name["ts_code"].isin(missing)]
print("缺失数量:", len(missing))
display(missing_df)

缺失数量: 5


,code,name,board,ts_code
463,001285,瑞立科密,深主板,001285.SZ
946,002445,中南文化,深主板,002445.SZ
3388,600673,东阳光,沪主板,600673.SH
3911,603056,德邦股份,沪主板,603056.SH
3970,603121,华培动力,沪主板,603121.SH


In [52]:
def build_full_history(stock_df, start_date="19901219", end_date="20260228"):
    trade_dates = get_trade_dates(start_date=start_date, end_date=end_date)

    for trade_date in tqdm(trade_dates, desc="全历史抓取"):
        try:
            one_day_df = fetch_one_day_all_stocks_fast(stock_df, trade_date)
            update_one_day_store(one_day_df)
        except Exception as e:
            print(f"{trade_date} 失败: {e}")

def get_trade_dates(start_date="19901219", end_date="20260228"):
    """
    用 AKShare 获取 A 股交易日历，返回 YYYYMMDD 字符串列表
    """
    cal = ak.tool_trade_date_hist_sina()
    cal["trade_date"] = pd.to_datetime(cal["trade_date"], errors="coerce")
    cal = cal.dropna(subset=["trade_date"])

    start_dt = pd.to_datetime(start_date, format="%Y%m%d")
    end_dt = pd.to_datetime(end_date, format="%Y%m%d")

    cal = cal[(cal["trade_date"] >= start_dt) & (cal["trade_date"] <= end_dt)].copy()
    cal = cal.sort_values("trade_date").reset_index(drop=True)

    return cal["trade_date"].dt.strftime("%Y%m%d").tolist()
    
# 用法
build_full_history(ts_name, start_date="19901219", end_date="20260228")


全历史抓取:   0%|          | 0/8589 [00:00<?, ?it/s]

当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新
当天没有数据可更新


KeyboardInterrupt: 

In [50]:
trade_dates = get_trade_dates("20260201", "20260228")
print(len(trade_dates))
print(trade_dates[:5])
print(trade_dates[-5:])

KeyboardInterrupt: 

In [48]:
cal = pro.trade_cal(exchange = "SSE",start_date='20180101', end_date='20181231')
print(cal.columns.tolist())
display(cal.head())

[]


""


In [56]:
daily_df = pro.daily(trade_date="20260224")
adj_df = pro.adj_factor(trade_date="20260224")

print("daily:", daily_df.shape)
print("adj:", adj_df.shape)
print("ts_name:", ts_name.shape)

daily: (0, 0)
adj: (0, 0)
ts_name: (3066, 4)


In [ ]:
from pathlib import Path

import pandas as pd
import tushare as ts
from tqdm.auto import tqdm

ts.set_token("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713d17a68")
pro = ts.pro_api("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713d17a68")

df = pro.daily(ts_code='000001.SZ', start_date='20180701', end_date='20180718')
df

""


In [17]:
test = pro.query("stock_basic", exchange="", list_status="L", fields="ts_code,symbol,name")
print(test.shape)
display(test.head())

(0, 0)


""
